# Paper plot generation

This notebook visualizes evaluation samples saved under `samples/sample_#/generated` and `samples/sample_#/gt`.

It produces one aligned multimodal timeline per sample:

1. `gt_video`
2. `gen_video`
3. `gt_audio_speak`
4. `gen_audio_speak`
5. `gt_audio_hear`
6. `gen_audio_hear`
7. `gt_mouse`
8. `gen_mouse`
9. `gt_key_press`
10. `gen_key_press`

The x-axis is time in seconds. Video frames are drawn at their exact 10 FPS temporal extent, mouse/key bins use the JSON `bin_ms`, and key rows use the decoded key names from `key_press.json`.


In [ ]:
from pathlib import Path

# ---- edit these ----
SAMPLES_ROOT = Path(
    "/ubc/cs/research/plai-scratch/mine01/plaicraft-model-pi0/logs/validation/runs/context_300/samples"
)

# Specify exactly which sample indices to plot.
SAMPLE_INDICES = [2]

# Latent/sample rates for each plotted modality.
VIDEO_FPS = 10.0
AUDIO_LATENT_HZ = 75.0
MOUSE_LATENT_HZ = 100.0
KEYPRESS_LATENT_HZ = 50.0

# Keypress binarization threshold. Saved JSON should already be 0/1, but this keeps
# the plotting robust to continuous values.
KEYPRESS_THRESHOLD = 0.5

# Figure/output settings.
SAVE_PNG = False
SAVE_PDF = True
DPI = 300
FIGURE_WIDTH_IN = 16.0

# Keep output inside notebooks/ even if this notebook is launched from repo root.
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "notebooks" / "paper_plot_generation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"SAMPLES_ROOT = {SAMPLES_ROOT}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")

In [ ]:
import json
import math
from collections import OrderedDict

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
from scipy.io import wavfile
from IPython.display import display, Image as IPyImage


# ----------------------------
# Generic IO helpers
# ----------------------------

def _normalize_audio_array(x: np.ndarray) -> np.ndarray:
    """Convert scipy wavfile output to mono float32 in [-1, 1] when possible."""
    x = np.asarray(x)
    if x.ndim == 2:
        x = x.mean(axis=1)

    if np.issubdtype(x.dtype, np.integer):
        info = np.iinfo(x.dtype)
        denom = max(abs(info.min), abs(info.max))
        x = x.astype(np.float32) / float(denom)
    else:
        x = x.astype(np.float32)

    return np.nan_to_num(x, copy=False)


def read_wav_mono(path: Path):
    path = Path(path)
    if not path.exists():
        return None

    sr, y = wavfile.read(str(path))
    y = _normalize_audio_array(y)
    return {"sr": int(sr), "y": y, "duration_s": len(y) / float(sr) if sr > 0 else 0.0}


def read_video_frames(path: Path, target_height_px: int | None = None):
    """Read all frames from an mp4 using OpenCV and return RGB frames."""
    path = Path(path)
    if not path.exists():
        return {"frames": [], "fps": VIDEO_FPS, "duration_s": 0.0}

    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")

    fps_from_file = cap.get(cv2.CAP_PROP_FPS)
    fps = float(fps_from_file) if fps_from_file and fps_from_file > 1e-6 else float(VIDEO_FPS)

    frames = []
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        if target_height_px is not None and frame_rgb.shape[0] > target_height_px:
            h, w = frame_rgb.shape[:2]
            new_w = max(1, int(round(w * (target_height_px / h))))
            frame_rgb = cv2.resize(frame_rgb, (new_w, target_height_px), interpolation=cv2.INTER_AREA)

        frames.append(frame_rgb)

    cap.release()
    return {"frames": frames, "fps": fps, "duration_s": len(frames) / fps if fps > 0 else 0.0}


# ----------------------------
# Mouse JSON helpers
# ----------------------------

def read_mouse_json(path: Path):
    """
    Supports both:
      1. parsed_decoded.series = [{time_ms, dx, dy}, ...]
      2. raw_decoded.frames = [{"dx": [...], "dy": [...]}, ...]
    """
    path = Path(path)
    if not path.exists():
        return {"time_s": np.array([]), "dx": np.array([]), "dy": np.array([]), "bin_ms": 10, "duration_s": 0.0}

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    bin_ms = int(obj.get("bin_ms", obj.get("parsed_decoded", {}).get("bin_ms", 10)))

    parsed = obj.get("parsed_decoded", {})
    series = parsed.get("series")
    if isinstance(series, list) and len(series) > 0:
        time_ms = np.array([float(row.get("time_ms", i * bin_ms)) for i, row in enumerate(series)], dtype=np.float32)
        dx = np.array([float(row.get("dx", 0.0)) for row in series], dtype=np.float32)
        dy = np.array([float(row.get("dy", 0.0)) for row in series], dtype=np.float32)
        duration_s = max(float(parsed.get("total_ms", 0)) / 1000.0, (time_ms[-1] + bin_ms) / 1000.0)
        return {"time_s": time_ms / 1000.0, "dx": dx, "dy": dy, "bin_ms": bin_ms, "duration_s": duration_s}

    raw_frames = obj.get("raw_decoded", {}).get("frames", [])
    dx_vals, dy_vals = [], []
    for fr in raw_frames:
        dx_vals.extend(fr.get("dx", []))
        dy_vals.extend(fr.get("dy", []))

    n = max(len(dx_vals), len(dy_vals))
    dx = np.asarray(dx_vals + [0] * (n - len(dx_vals)), dtype=np.float32)
    dy = np.asarray(dy_vals + [0] * (n - len(dy_vals)), dtype=np.float32)
    time_s = np.arange(n, dtype=np.float32) * (bin_ms / 1000.0)
    return {"time_s": time_s, "dx": dx, "dy": dy, "bin_ms": bin_ms, "duration_s": n * bin_ms / 1000.0}


# ----------------------------
# Keypress JSON helpers
# ----------------------------

def read_keypress_json(path: Path):
    """
    Supports both:
      1. raw_decoded.frames = [{key_name: [0/1 per bin], ...}, ...]
      2. parsed_decoded.events = [{key_name, start_ms, end_ms}, ...]
    Returns a key-name matrix shaped [num_keys, num_time_bins].
    """
    path = Path(path)
    if not path.exists():
        return {
            "key_names": [],
            "matrix": np.zeros((0, 0), dtype=np.float32),
            "bin_ms": 10,
            "duration_s": 0.0,
        }

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    bin_ms = int(obj.get("bin_ms", obj.get("parsed_decoded", {}).get("bin_ms", 10)))
    raw_frames = obj.get("raw_decoded", {}).get("frames", [])

    if isinstance(raw_frames, list) and len(raw_frames) > 0 and isinstance(raw_frames[0], dict):
        # Preserve JSON order from the first frame, then append any late-appearing keys.
        key_names = list(raw_frames[0].keys())
        seen = set(key_names)
        for fr in raw_frames[1:]:
            for k in fr.keys():
                if k not in seen:
                    key_names.append(k)
                    seen.add(k)

        bins_by_key = OrderedDict((k, []) for k in key_names)
        for fr in raw_frames:
            bins_per_frame = None
            if len(fr) > 0:
                first_val = next(iter(fr.values()))
                bins_per_frame = len(first_val) if isinstance(first_val, list) else 1

            for k in key_names:
                vals = fr.get(k)
                if vals is None:
                    vals = [0] * int(bins_per_frame or 0)
                elif not isinstance(vals, list):
                    vals = [vals]
                bins_by_key[k].extend(vals)

        matrix = np.asarray([bins_by_key[k] for k in key_names], dtype=np.float32)
        n_bins = matrix.shape[1] if matrix.ndim == 2 else 0
        return {
            "key_names": key_names,
            "matrix": matrix,
            "bin_ms": bin_ms,
            "duration_s": n_bins * bin_ms / 1000.0,
        }

    parsed = obj.get("parsed_decoded", {})
    events = parsed.get("events", [])
    if isinstance(events, list) and len(events) > 0:
        key_names = []
        for ev in events:
            k = ev.get("key_name", "unknown")
            if k not in key_names:
                key_names.append(k)

        total_ms = parsed.get("total_ms")
        if total_ms is None:
            total_ms = max(float(ev.get("end_ms", 0.0)) for ev in events)
        n_bins = max(1, int(math.ceil(float(total_ms) / bin_ms)))
        matrix = np.zeros((len(key_names), n_bins), dtype=np.float32)

        key_to_idx = {k: i for i, k in enumerate(key_names)}
        for ev in events:
            k = ev.get("key_name", "unknown")
            start_bin = int(math.floor(float(ev.get("start_ms", 0.0)) / bin_ms))
            end_bin = int(math.ceil(float(ev.get("end_ms", 0.0)) / bin_ms))
            matrix[key_to_idx[k], max(0, start_bin):max(0, end_bin)] = 1.0

        return {
            "key_names": key_names,
            "matrix": matrix,
            "bin_ms": bin_ms,
            "duration_s": n_bins * bin_ms / 1000.0,
        }

    total_ms = float(parsed.get("total_ms", 0.0))
    return {
        "key_names": [],
        "matrix": np.zeros((0, 0), dtype=np.float32),
        "bin_ms": bin_ms,
        "duration_s": total_ms / 1000.0,
    }


def active_key_union(gt_key, gen_key, threshold: float = KEYPRESS_THRESHOLD):
    """Union of active keys across gt/gen, preserving decoded key-name order."""
    active = []
    seen = set()

    for data in (gt_key, gen_key):
        names = data["key_names"]
        mat = data["matrix"]
        if mat.size == 0:
            continue
        is_active = mat.max(axis=1) >= threshold
        for name, on in zip(names, is_active):
            if bool(on) and name not in seen:
                active.append(name)
                seen.add(name)

    return active


# ----------------------------
# Sample loading
# ----------------------------

def sample_dir_for_index(samples_root: Path, sample_index: int) -> Path:
    return Path(samples_root) / f"sample_{int(sample_index)}"


def load_sample_modalities(samples_root: Path, sample_index: int):
    sample_dir = sample_dir_for_index(samples_root, sample_index)
    if not sample_dir.exists():
        raise FileNotFoundError(f"Sample directory not found: {sample_dir}")

    out = {"sample_index": int(sample_index), "sample_dir": sample_dir}
    for split in ("gt", "generated"):
        d = sample_dir / split
        out[split] = {
            "video": read_video_frames(d / "video.mp4"),
            "audio_speak": read_wav_mono(d / "audio_speak.wav"),
            "audio_hear": read_wav_mono(d / "audio_hear.wav"),
            "mouse": read_mouse_json(d / "mouse_movement.json"),
            "key_press": read_keypress_json(d / "key_press.json"),
        }

    duration_s = 0.0
    for split in ("gt", "generated"):
        for item in out[split].values():
            if item is not None:
                duration_s = max(duration_s, float(item.get("duration_s", 0.0)))

    # Avoid zero-width axes.
    out["duration_s"] = max(duration_s, 1e-3)
    return out


In [ ]:
# ----------------------------
# Plotting helpers
# ----------------------------

def _format_axes_common(ax, duration_s: float, label: str):
    ax.set_xlim(0.0, duration_s)
    ax.text(
        0.006, 0.95, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=10, fontweight="bold",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5),
        zorder=20,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _add_latent_frequency_ticks(ax, duration_s: float, hz: float, tick_height: float = 0.075):
    """Draw small latent/sample-rate ticks that visibly overlap the bottom spine."""
    hz = float(hz)
    if hz <= 0 or duration_s <= 0:
        return

    dt = 1.0 / hz
    ticks = np.arange(0.0, duration_s + 0.5 * dt, dt)
    ticks = ticks[(ticks >= 0.0) & (ticks <= duration_s + 1e-9)]

    # Start slightly below the axes boundary and draw above the spine.
    # This removes the 1-pixel antialiasing gap between the tick and the black axis line.
    tick_lines = ax.vlines(
        ticks, -0.035, tick_height,
        transform=ax.get_xaxis_transform(),  # x=data coords, y=axes-fraction coords
        color="0.55", linewidth=0.5, alpha=1.0,
        zorder=100, clip_on=False,
    )
    tick_lines.set_capstyle("projecting")


def _video_slot_height_data_units(ax, duration_s: float, slot_width_s: float, frame_shape, renderer):
    """Compute the row-height fraction needed to preserve the original frame aspect ratio."""
    h, w = frame_shape[:2]
    bbox = ax.get_window_extent(renderer=renderer)
    ax_w_px = max(float(bbox.width), 1.0)
    ax_h_px = max(float(bbox.height), 1.0)
    desired_aspect = float(w) / float(h)  # width / height

    # Solve for the image height in axis data units so the displayed image aspect ratio
    # matches the original frame aspect ratio while its width stays truthful in time.
    slot_height = (slot_width_s * ax_w_px) / (max(duration_s, 1e-6) * ax_h_px * desired_aspect)
    return min(max(slot_height, 1e-6), 1.0)


def _plot_video_row(ax, video_data, duration_s: float, label: str, fig, fps: float = VIDEO_FPS):
    frames = video_data["frames"]
    fps = float(fps or video_data.get("fps", VIDEO_FPS) or VIDEO_FPS)
    frame_dt = 1.0 / fps

    if len(frames) == 0:
        ax.text(0.5, 0.5, "missing video", ha="center", va="center", transform=ax.transAxes)
    else:
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()

        for i, frame in enumerate(frames):
            t0 = i * frame_dt
            t1 = min((i + 1) * frame_dt, duration_s)
            if t0 >= duration_s:
                break

            slot_width_s = max(t1 - t0, 1e-6)
            slot_height = _video_slot_height_data_units(ax, duration_s, slot_width_s, frame.shape, renderer)
            y0 = 0.5 - slot_height / 2.0
            y1 = 0.5 + slot_height / 2.0

            ax.imshow(
                frame,
                extent=(t0, t1, y0, y1),
                interpolation="nearest",
                aspect="auto",
                zorder=1,
            )

    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([])
    _format_axes_common(ax, duration_s, label)
    _add_latent_frequency_ticks(ax, duration_s, fps)


def _thin_signal(t, y, max_points=8000):
    if t is None or y is None or len(t) <= max_points:
        return t, y
    stride = int(math.ceil(len(t) / max_points))
    return t[::stride], y[::stride]


def _plot_audio_row(ax, audio_data, duration_s: float, label: str, y_abs_max: float):
    if audio_data is None or len(audio_data["y"]) == 0:
        ax.text(0.5, 0.5, "missing audio", ha="center", va="center", transform=ax.transAxes)
    else:
        sr = audio_data["sr"]
        y = audio_data["y"]
        t = np.arange(len(y), dtype=np.float32) / float(sr)
        t_plot, y_plot = _thin_signal(t, y)
        ax.plot(t_plot, y_plot, linewidth=0.7)

    y_abs_max = max(float(y_abs_max), 1e-6)
    ax.set_ylim(-y_abs_max, y_abs_max)
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    _format_axes_common(ax, duration_s, label)
    _add_latent_frequency_ticks(ax, duration_s, AUDIO_LATENT_HZ)


def _plot_mouse_row(ax, mouse_data, duration_s: float, label: str, y_abs_max: float):
    t = mouse_data["time_s"]
    dx = mouse_data["dx"]
    dy = mouse_data["dy"]

    if len(t) == 0:
        ax.text(0.5, 0.5, "missing mouse", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.step(t, dx, where="post", linewidth=0.9, label="dx")
        ax.step(t, dy, where="post", linewidth=0.9, label="dy")
        ax.legend(loc="upper right", fontsize=8, frameon=False, ncol=2)

    y_abs_max = max(float(y_abs_max), 1.0)
    ax.set_ylim(-y_abs_max, y_abs_max)
    ax.grid(axis="x", alpha=0.25)
    _format_axes_common(ax, duration_s, label)
    _add_latent_frequency_ticks(ax, duration_s, MOUSE_LATENT_HZ)


def _select_key_matrix(key_data, key_names):
    """Return matrix rows in the requested key-name order, filling absent keys with zeros."""
    source_names = key_data["key_names"]
    source_matrix = key_data["matrix"]
    n_bins = source_matrix.shape[1] if source_matrix.ndim == 2 else 0
    name_to_idx = {name: i for i, name in enumerate(source_names)}

    rows = []
    for name in key_names:
        idx = name_to_idx.get(name)
        if idx is None:
            rows.append(np.zeros((n_bins,), dtype=np.float32))
        else:
            rows.append(source_matrix[idx])

    if len(rows) == 0:
        return np.zeros((0, n_bins), dtype=np.float32)
    return np.stack(rows, axis=0)


def _plot_keypress_row(ax, key_data, key_names, duration_s: float, label: str, threshold: float = KEYPRESS_THRESHOLD):
    mat = _select_key_matrix(key_data, key_names)

    if len(key_names) == 0:
        ax.text(0.5, 0.5, "no active keys", ha="center", va="center", transform=ax.transAxes)
        ax.set_yticks([])
    elif mat.shape[1] == 0:
        ax.text(0.5, 0.5, "missing keypress", ha="center", va="center", transform=ax.transAxes)
        ax.set_yticks(np.arange(len(key_names)))
        ax.set_yticklabels(key_names, fontsize=8)
    else:
        bin_ms = int(key_data.get("bin_ms", 10))
        key_duration_s = mat.shape[1] * bin_ms / 1000.0
        mat_bin = (mat >= threshold).astype(np.float32)
        ax.imshow(
            mat_bin,
            aspect="auto",
            interpolation="nearest",
            origin="lower",
            cmap="Greys",
            extent=(0.0, key_duration_s, -0.5, len(key_names) - 0.5),
            vmin=0.0,
            vmax=1.0,
        )
        ax.set_yticks(np.arange(len(key_names)))
        ax.set_yticklabels(key_names, fontsize=8)

    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    _format_axes_common(ax, duration_s, label)
    _add_latent_frequency_ticks(ax, duration_s, KEYPRESS_LATENT_HZ)


def _audio_group_absmax(*items):
    vals = []
    for item in items:
        if item is not None and len(item["y"]) > 0:
            vals.append(float(np.max(np.abs(item["y"]))))
    return max(vals + [1e-4])


def _mouse_pair_absmax(a, b):
    vals = []
    for item in (a, b):
        if item is not None:
            if len(item["dx"]) > 0:
                vals.append(float(np.max(np.abs(item["dx"]))))
            if len(item["dy"]) > 0:
                vals.append(float(np.max(np.abs(item["dy"]))))
    return max(vals + [1.0])


def plot_sample_timeline(
    sample_index: int,
    samples_root: Path = SAMPLES_ROOT,
    output_dir: Path = OUTPUT_DIR,
    save_png: bool = SAVE_PNG,
    save_pdf: bool = SAVE_PDF,
    dpi: int = DPI,
    figure_width_in: float = FIGURE_WIDTH_IN,
    show: bool = True,
):
    data = load_sample_modalities(samples_root, sample_index)
    duration_s = data["duration_s"]

    gt = data["gt"]
    gen = data["generated"]

    active_keys = active_key_union(gt["key_press"], gen["key_press"], threshold=KEYPRESS_THRESHOLD)

    # Dynamic row heights: keep video reasonably tall and make key rows taller when many keys are active.
    key_ratio = max(0.9, min(3.0, 0.22 * max(1, len(active_keys))))
    video_ratio = max(1.35, 1.35 * (1.0 / max(duration_s, 1.0 / VIDEO_FPS)))
    height_ratios = [
        video_ratio, video_ratio,    # video
        0.72, 0.72,    # audio_speak
        0.72, 0.72,    # audio_hear
        0.78, 0.78,    # mouse
        key_ratio, key_ratio,
    ]
    figure_height_in = max(8.5, sum(height_ratios) * 0.85)

    fig, axes = plt.subplots(
        10, 1,
        figsize=(figure_width_in, figure_height_in),
        sharex=True,
        gridspec_kw={"height_ratios": height_ratios, "hspace": 0.16},
        constrained_layout=False,
    )
    fig.subplots_adjust(left=0.13, right=0.995, top=0.99, bottom=0.055)

    _plot_video_row(axes[0], gt["video"], duration_s, "gt_video", fig=fig, fps=VIDEO_FPS)
    _plot_video_row(axes[1], gen["video"], duration_s, "gen_video", fig=fig, fps=VIDEO_FPS)

    audio_absmax = _audio_group_absmax(
        gt["audio_speak"], gen["audio_speak"], gt["audio_hear"], gen["audio_hear"]
    )
    _plot_audio_row(axes[2], gt["audio_speak"], duration_s, "gt_audio_speak", audio_absmax)
    _plot_audio_row(axes[3], gen["audio_speak"], duration_s, "gen_audio_speak", audio_absmax)
    _plot_audio_row(axes[4], gt["audio_hear"], duration_s, "gt_audio_hear", audio_absmax)
    _plot_audio_row(axes[5], gen["audio_hear"], duration_s, "gen_audio_hear", audio_absmax)

    mouse_absmax = _mouse_pair_absmax(gt["mouse"], gen["mouse"])
    _plot_mouse_row(axes[6], gt["mouse"], duration_s, "gt_mouse", mouse_absmax)
    _plot_mouse_row(axes[7], gen["mouse"], duration_s, "gen_mouse", mouse_absmax)

    _plot_keypress_row(axes[8], gt["key_press"], active_keys, duration_s, "gt_key_press")
    _plot_keypress_row(axes[9], gen["key_press"], active_keys, duration_s, "gen_key_press")

    # Shared x-axis formatting at 5 FPS: 0, 0.2, 0.4, ...
    axes[-1].set_xlabel("Time (s)", fontsize=11)
    axes[-1].xaxis.set_major_locator(ticker.MultipleLocator(0.2))
    axes[-1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:g}"))
    for ax in axes[:-1]:
        ax.tick_params(axis="x", labelbottom=False)

    sample_dir = data["sample_dir"]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_base = output_dir / f"{sample_dir.name}_multimodal_timeline"

    saved_paths = []
    if save_png:
        png_path = out_base.with_suffix(".png")
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        saved_paths.append(png_path)

    if save_pdf:
        pdf_path = out_base.with_suffix(".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        saved_paths.append(pdf_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return saved_paths


def generate_plots_for_samples(sample_indices=SAMPLE_INDICES, show=True):
    all_paths = {}
    for idx in sample_indices:
        print(f"Generating sample_{idx} ...")
        paths = plot_sample_timeline(idx, show=show)
        all_paths[int(idx)] = paths
        for p in paths:
            print(f"  saved: {p}")
    return all_paths



## Generate plots

Edit `SAMPLE_INDICES` in the first code cell to choose exactly which `sample_#` folders to render.


In [ ]:
saved = generate_plots_for_samples(SAMPLE_INDICES, show=True)
saved

## Optional: preview a saved PNG

This previews the first generated PNG inside the notebook. It is useful when running headless on a cluster notebook server.


In [ ]:
first_png = next((p for paths in saved.values() for p in paths if p.suffix == ".png"), None)
if first_png is not None:
    display(IPyImage(filename=str(first_png)))
else:
    print("No PNG was generated.")

In [ ]:
# Concatenate all generated audio into two big wav files, one for speak and one for hear.
sample_root = Path(SAMPLES_ROOT)
sample_dirs = sorted(
    [p for p in sample_root.iterdir() if p.is_dir() and (p / "generated").exists()],
    key=lambda p: p.name,
)

if not sample_dirs:
    raise FileNotFoundError(f"No sample directories with generated audio found under {sample_root}")

speak_chunks = []
hear_chunks = []
output_sr = None

for sample_dir in sample_dirs:
    speak = read_wav_mono(sample_dir / "generated" / "audio_speak.wav")
    hear = read_wav_mono(sample_dir / "generated" / "audio_hear.wav")
    if speak is None or hear is None:
        raise FileNotFoundError(f"Missing generated audio in {sample_dir / 'generated'}")

    if output_sr is None:
        output_sr = int(speak["sr"])

    if int(speak["sr"]) != output_sr or int(hear["sr"]) != output_sr:
        raise ValueError(
            f"Sample rate mismatch in {sample_dir}: speak={speak['sr']}, hear={hear['sr']}, expected={output_sr}"
        )

    speak_chunks.append(speak["y"])
    hear_chunks.append(hear["y"])

speak_concat = np.concatenate(speak_chunks, axis=0).astype(np.float32, copy=False)
hear_concat = np.concatenate(hear_chunks, axis=0).astype(np.float32, copy=False)

output_root = sample_root.parent
speak_out_path = output_root / "generated_audio_speak_concat.wav"
hear_out_path = output_root / "generated_audio_hear_concat.wav"

wavfile.write(str(speak_out_path), output_sr, speak_concat)
wavfile.write(str(hear_out_path), output_sr, hear_concat)

print(f"Wrote {speak_out_path} ({len(speak_chunks)} clips, {speak_concat.size / float(output_sr):.2f}s)")
print(f"Wrote {hear_out_path} ({len(hear_chunks)} clips, {hear_concat.size / float(output_sr):.2f}s)")

Wrote /ubc/cs/research/plai-scratch/mine01/plaicraft-model-pi0/logs/validation/runs/context_300/samples/generated_audio_speak_concat.wav (1000 clips, 1000.00s)
Wrote /ubc/cs/research/plai-scratch/mine01/plaicraft-model-pi0/logs/validation/runs/context_300/samples/generated_audio_hear_concat.wav (1000 clips, 1000.00s)


## Validation metrics across context sizes

Plot metrics from validation runs with different context window sizes (10, 100, 300, 600, 1200).


In [ ]:
# Load validation metrics across context sizes
import json

VALIDATION_RUNS_ROOT = Path("/ubc/cs/research/plai-scratch/mine01/plaicraft-model-pi0/logs/validation/runs")
CONTEXT_SIZES = [10, 100, 300, 600, 1200]

def load_validation_report(context_size: int):
    """Load validation_report.json for a given context size."""
    report_path = VALIDATION_RUNS_ROOT / f"context_{context_size}" / "validation_report.json"
    if not report_path.exists():
        return None
    with open(report_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Load all reports
reports = {}
for ctx_size in CONTEXT_SIZES:
    reports[ctx_size] = load_validation_report(ctx_size)

print(f"Loaded reports for context sizes: {list(reports.keys())}")
print(f"Available metrics keys: {list(reports[CONTEXT_SIZES[0]].keys()) if reports[CONTEXT_SIZES[0]] else 'None'}")


In [ ]:
# Define metric configurations with names and directionality
METRIC_CONFIGS = {
    "video": {
        "psnr": {"name": "PSNR", "higher_better": True},
        "ssim": {"name": "SSIM", "higher_better": True},
        "lpips": {"name": "LPIPS", "higher_better": False},
        "fid": {"name": "FID", "higher_better": False},
    },
    "audio_speaking": {
        "frechet_w2v2": {"name": "Audio Speaking Fréchet w2v2", "higher_better": False},
    },
    "audio_hearing": {
        "frechet_w2v2": {"name": "Audio Hearing Fréchet w2v2", "higher_better": False},
    },
    "keypress": {
        "normalized_hamming": {"name": "Keypress Normalized Hamming Distance", "higher_better": False},
    },
    "mouse": {
        "IDE_mean": {"name": "Mouse IDE (Mean)", "higher_better": False},
        "ADE_mean": {"name": "Mouse ADE (Mean)", "higher_better": False},
        "FDE_mean": {"name": "Mouse FDE (Mean)", "higher_better": False},
        "PLD_mean": {"name": "Mouse PLD (Mean)", "higher_better": False},
    },
}

def get_metric_value(report, metric_type: str, metric_key: str):
    """Extract metric value from a validation report."""
    if not report:
        return None
    
    if metric_type == "video":
        return report.get("video", {}).get(metric_key)
    elif metric_type == "audio_speaking":
        return report.get("audio", {}).get("speaking", {}).get(metric_key)
    elif metric_type == "audio_hearing":
        return report.get("audio", {}).get("hearing", {}).get(metric_key)
    elif metric_type == "keypress":
        if metric_key == "normalized_hamming":
            return report.get("keypress_mouseclick", {}).get("normalized_hamming")
    elif metric_type == "mouse":
        return report.get("mouse_movement", {}).get(metric_key)
    return None

# Extract data for each metric
metrics_data = {}
for metric_type, metrics in METRIC_CONFIGS.items():
    metrics_data[metric_type] = {}
    for metric_key, config in metrics.items():
        values = []
        valid_contexts = []
        for ctx_size in CONTEXT_SIZES:
            val = get_metric_value(reports[ctx_size], metric_type, metric_key)
            if val is not None:
                values.append(float(val))
                valid_contexts.append(ctx_size)
        if values:
            metrics_data[metric_type][metric_key] = {
                "values": np.array(values),
                "contexts": np.array(valid_contexts),
                "config": config,
            }

print("Extracted metrics data successfully")
for metric_type in metrics_data:
    print(f"  {metric_type}: {len(metrics_data[metric_type])} metrics")


In [ ]:
# Plot all metrics
def plot_metrics_across_contexts():
    """Create line plots for each metric across context sizes."""
    output_dir_metrics = OUTPUT_DIR / "context_metrics"
    output_dir_metrics.mkdir(parents=True, exist_ok=True)
    
    plot_count = 0
    
    for metric_type in sorted(metrics_data.keys()):
        for metric_key in sorted(metrics_data[metric_type].keys()):
            data = metrics_data[metric_type][metric_key]
            config = data["config"]
            contexts = data["contexts"]
            values = data["values"]
            
            # Create figure
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Plot line
            ax.plot(contexts, values, marker='o', linewidth=2.5, markersize=8, color='steelblue')
            ax.grid(True, alpha=0.3)
            
            # Labels and title
            metric_name = config["name"]
            higher_better = config["higher_better"]
            direction_text = "↑ Higher is better" if higher_better else "↓ Lower is better"
            
            ax.set_xlabel("Context Size (frames)", fontsize=12, fontweight='bold')
            ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
            ax.set_title(f"{metric_name} vs Context Size\n{direction_text}", 
                        fontsize=13, fontweight='bold', pad=15)
            
            # Set x-axis to show all context sizes
            ax.set_xticks(CONTEXT_SIZES)
            ax.set_xscale('log')
            
            # Add value labels on points
            for ctx, val in zip(contexts, values):
                ax.text(ctx, val, f'{val:.4g}', ha='center', va='bottom', fontsize=9)
            
            plt.tight_layout()
            
            # Save
            filename = f"{metric_type}_{metric_key}"
            for fmt, ext in [("png", ".png"), ("pdf", ".pdf")]:
                if fmt == "png" and SAVE_PNG:
                    path = output_dir_metrics / f"{filename}.png"
                    fig.savefig(path, dpi=DPI, bbox_inches="tight")
                elif fmt == "pdf" and SAVE_PDF:
                    path = output_dir_metrics / f"{filename}.pdf"
                    fig.savefig(path, bbox_inches="tight")
            
            plt.close(fig)
            plot_count += 1
    
    print(f"Generated {plot_count} metric plots in {output_dir_metrics}")
    return output_dir_metrics

context_metrics_dir = plot_metrics_across_contexts()


In [ ]:
# Create a summary table of all metrics
import pandas as pd

def create_metrics_summary_table():
    """Create a DataFrame with all metrics for easy comparison."""
    rows = []
    
    for metric_type in sorted(metrics_data.keys()):
        for metric_key in sorted(metrics_data[metric_type].keys()):
            data = metrics_data[metric_type][metric_key]
            config = data["config"]
            contexts = data["contexts"]
            values = data["values"]
            
            metric_name = config["name"]
            direction = "Higher ↑" if config["higher_better"] else "Lower ↓"
            
            row = {
                "Metric Type": metric_type,
                "Metric Name": metric_name,
                "Direction": direction,
            }
            
            # Add values for each context size
            for ctx, val in zip(contexts, values):
                row[f"Context {int(ctx)}"] = f"{val:.6f}"
            
            rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

summary_df = create_metrics_summary_table()
print("\nMetrics Summary Table:")
print(summary_df.to_string())

# Optionally save to CSV
csv_path = OUTPUT_DIR / "context_metrics_summary.csv"
summary_df.to_csv(csv_path, index=False)
print(f"\nSummary table saved to: {csv_path}")


## Overview

The above code generates individual line plots for each metric, showing how performance changes with different context window sizes:

- **Video metrics**: PSNR, SSIM (↑ higher is better), LPIPS, FID (↓ lower is better)
- **Audio metrics**: Fréchet w2v2 distance for speaking and hearing (↓ lower is better)
- **Keypress**: Normalized Hamming distance (↓ lower is better)
- **Mouse movement**: IDE, ADE, FDE, and PLD means (↓ lower is better)

Each plot includes:
- X-axis: Context size (log scale) showing 10, 100, 300, 600, 1200 frames
- Y-axis: Metric value
- Direction indicator (↑/↓) showing whether higher or lower performance is better
- Value labels on each data point

All plots are saved to `notebooks/paper_plot_generation_outputs/context_metrics/` as PNG and PDF files, and a summary CSV is also generated.
